In [ ]:
import numpy as np
import random
import importlib
import Player_Baselines
import Poker_Agent
from Player import HumanPlayer
import matplotlib.pyplot as plt
importlib.reload(Poker_Agent)
importlib.reload(Player_Baselines)
from Player_Baselines import FixedOpponent
from Poker_Agent import QLearningPlayer
from itertools import cycle
from pathlib import Path
import sys


In [ ]:
class SimpleLeDucPoker:
    def __init__(self):
        # Tiny deck
        self.deck = ['J', 'J', 'Q', 'Q', 'K', 'K']
        # Possible actions
        self.actions = ['check', 'call', 'raise', 'fold']
        self.win = 0
        self.reset()

    def reset(self):
        # Shuffle cards
        self.deck = ['J', 'J', 'Q', 'Q', 'K', 'K']
        random.shuffle(self.deck)

        # Deal private cards
        self.player_card = self.deck.pop()
        self.opponent_card = self.deck.pop()

        # Public card starts hidden
        self.public_card = None
        self.first_round = 0
        self.end_of_turn = 0 
        self.raised = False
        self.folded = False
        self.end_of_phase = False
        self.history = []
        self.record = []

    def log_action(self, actor, action, player, opponent):
        self.history.append({
            'actor': actor,
            'action': action,
            'player_bet': player.bet,
            'opponent_bet': opponent.bet
        })

    def get_chips(self, actor, player, opponent):
        return player.chips if actor == 'player' else opponent.chips

    def get_legal_actions(self, actor, player, opponent):
        if actor == 'player':
            if self.first_round==0 and self.public_card is None:
                return ["call", "raise", "fold"]
            elif opponent.raised and not player.raised:
                return ["call", "raise", "fold"]
            elif not opponent.raised and not player.raised and self.first_round<=2:
                return ["check", "raise", "fold"]
            else:
                return ["call", "fold"]
        else:
            if self.first_round==0 and self.public_card is None:
                return ["call", "raise", "fold"]
            elif not opponent.raised and player.raised:
                return ["call", "raise", "fold"]
            elif not opponent.raised and not player.raised and self.first_round<=2:
                return ["check", "raise", "fold"]
            else:
                return ["call", "fold"]

    # Big blind - Small blind
    def initial_bet(self,player,opponent):
        if self.turn == "player":
            player.bet = 1
            opponent.bet=2
        else:
            opponent.bet=1
            player.bet = 2

    # Match the other player's bet
    def call_amount(self, actor, player, opponent):
        if actor == 'player':
            return max(opponent.bet - player.bet, 0)
        return max(player.bet - opponent.bet, 0)

    # Raise the bet by a fixed amount (+2 in the first round, +4 in the second round)
    def raise_amount(self, actor, player, opponent):
        if self.public_card is None:
            return self.call_amount(actor, player, opponent) + 2
        else:
            return self.call_amount(actor, player, opponent) + 4


    def step(self, action, player, opponent, player_starts):
        if self.turn == "player":                
            # print("Player action:", action, "\n")
            act=self.apply_action("player", action, player, opponent)
            self.first_round += 1
            self.log_action("Player", action, player, opponent)

            if (act==True):
                # print("Player folded. Opponent wins the pot.")
                return  -player.bet, True
            
            if (self.end_of_turn >= 2):
                if self.public_card is not None:
                    # print("Showdown! Player card:", self.player_card, "Opponent card:", self.opponent_card, "Public card:", self.public_card)
                    return  self.evaluate_winner(player, opponent), True
                self.end_of_turn = 0
                if self.public_card is None:
                    self.public_card = self.deck.pop()
                self.first_round = 0
                self.turn = player_starts
                self.record.clear()
                return  0, True

            self.turn = "opponent"
            return  0, False
        else:
            # print("Opponent action:", action, "\n")
            act=self.apply_action("opponent", action, player, opponent)
            self.first_round += 1
            self.log_action("Opponent", action, player, opponent)

            if (act==True):
                # print("Opponent folded. Player wins the pot.")
                return  opponent.bet, True
            
            if (self.end_of_turn >= 2):
                if self.public_card is not None:
                    # print("Showdown! Player card:", self.player_card, "Opponent card:", self.opponent_card, "Public card:", self.public_card)
                    return  self.evaluate_winner(player, opponent), True
                self.end_of_turn = 0
                if self.public_card is None:
                    self.public_card = self.deck.pop()
                self.first_round = 0
                self.turn = player_starts
                self.record.clear()
                return 0, True

            self.turn = "player"
            return 0 , False


    def apply_action(self, actor, action, player, opponent):

        if action == "raise":
            amount = self.raise_amount(actor, player, opponent)
            self.end_of_turn = 0
            if actor == "player":
                player.raised = True
            else:
                opponent.raised = True
            self.record.append('r')

        elif action == "call":
            if self.first_round != 0:
                self.end_of_turn += 2
            amount = self.call_amount(actor, player, opponent)
            self.record.append('c')

        elif action == "fold":
            self.end_of_phase = True
            self.folded = True
            return True
        
        else:
            if self.first_round == 1:
                self.end_of_turn += 2
            else:
                self.end_of_turn += 1
            self.record.append('c')
            return False

        if actor == "player":
            player.bet += amount
        else:
            opponent.bet += amount
        return False
    
    # Pair wins. If there's no pair, high card wins.
    def evaluate_winner(self, player, opponent):
        player_pair = (self.player_card == self.public_card)
        opponent_pair = (self.opponent_card == self.public_card)

        if player_pair and not opponent_pair:
            return opponent.bet

        if opponent_pair and not player_pair:
            return -player.bet

        values = {'J': 1, 'Q': 2, 'K': 3}

        if values[self.player_card] > values[self.opponent_card]:
            return opponent.bet

        if values[self.player_card] < values[self.opponent_card]:
            return -player.bet
        return 0


In [ ]:
def training_game(env, player_starts, player, opponent):
    """
    Args:
        env: SimpleLeDucPoker instance
        player_starts: True if player acts first
        player: QLearningPlayer instance (or any player)
        opponent: PokerPlayer instance
    
    Returns:
        reward: chip change for the player
    """
    env.reset()    
    # print("\n-------------------New game started-------------------")  
    env.turn = player_starts
    # print("-------------------"+ player_starts +" starts-------------------\n")
    # print("Player card is: ", env.player_card)
    end_of_phase = False
    
    for i in range(2):
        
        if i == 0:
            env.initial_bet(player, opponent)
        
        if (i == 1 and not env.folded):
            # print("Phase 2: Deal private cards and initial betting\n")
            # print("Public card is: ", env.public_card)
            end_of_phase = False
            player.raised = False
            opponent.raised = False
            
        while not end_of_phase:
            legal_actions = env.get_legal_actions(env.turn, player, opponent)
            history = ''.join(env.record)

            if env.turn == "player":
                if i == 0:  # Round 1
                    action = player.action_1(env.player_card, legal_actions, history, player_starts)
                else:       # Round 2
                    action = player.action_2(env.player_card, legal_actions, history, player_starts, env.public_card)
            else:  # opponent's turn
                if i == 0:  # Round 1
                    if isinstance(opponent, QLearningPlayer):
                        action = opponent.action_1(env.opponent_card, legal_actions, history, player_starts)
                    else:
                        action = opponent.action_1(env.opponent_card, legal_actions, player_starts)
                else:       # Round 2
                    if isinstance(opponent, QLearningPlayer):
                        action = opponent.action_2(env.player_card, legal_actions, history, player_starts, env.public_card)
                    else:
                        action = opponent.action_2(env.opponent_card, legal_actions, player_starts, env.public_card)      
            reward, end_of_phase = env.step(action, player, opponent, player_starts)
            
            if env.folded or (end_of_phase and i == 1):
                player.learn(player.last_state, player.last_action, reward, next_state=None, done=True)
            elif env.turn == "player":
                new_history = ''.join(env.record)
                if not end_of_phase:
                    player.next_state = player.encode_state(env.player_card, env.public_card, new_history, 'SB' if player_starts=="player" else 'BB')
                else:
                    player.next_state = None
                player.learn(player.last_state, player.last_action, reward, player.next_state, end_of_phase)
            if end_of_phase:
                break
    
    # Return reward in terms of Big Blinds, not chips (Big Blind = 2 chips)
    # print("\nFinal Reward:  ", reward/2)
    return reward/2

In [ ]:

# Agent training for agents meant to play against fixed playing styles

def train_agent(games, opponent_style, save_path):
    """Train Q-learning agent against a baseline opponent - Independent games approach"""
    player = QLearningPlayer(model_path=None, alpha=0.2, gamma=0.95, epsilon=1.0)  # Initial epsilon - drops during training 
    if opponent_style in ["Aggressive", "Tight"]:
        games += 100000
    players = ['player', 'opponent']
    player_cycle = cycle(players)
    cumulative_rewards = np.zeros(games)
    game_rewards = np.zeros(games)
    env = SimpleLeDucPoker()

    for game_idx in range(games):

        # Opponent style is fixed. If "Mixed", opponent style is any of the fixed styles with uniform probability
        if opponent_style != "Mixed":
            opponent = FixedOpponent(opponent_style)
        else:
            rand_style = np.random.choice(['Pal-Bot', 'Random', 'Tight', 'Limping', 'Aggressive', 'Honest-strength'])
            opponent = FixedOpponent(rand_style)

        player_starts = next(player_cycle)
        reward = training_game(env, player_starts, player, opponent)

        game_rewards[game_idx] = reward
        cumulative_rewards[game_idx] = cumulative_rewards[game_idx-1] + reward if game_idx > 0 else reward
        
        player.reset()
        env.reset()

        # epsilon decays at the end of each training episode instead of each step (which was the case when it decayed within player.learn())
        player.epsilon = max(0.01, 0.99999*player.epsilon)

        
    player.save_model(save_path)
    np.savetxt(f"./data/training/agent_vs_{opponent_style}.csv", cumulative_rewards)

    plt.figure()
    plt.title(f"Agent's Reward vs {opponent_style} Opponent")
    plt.xlabel("Game Number")
    plt.ylabel("Cumulative Score")
    plt.plot(np.arange(1, games+1), cumulative_rewards, label="Cumulative Reward")
    plt.grid()
    plt.legend()
    plt.savefig(f'./data/images/training_vs_{opponent_style}.jpg')
    plt.show()


In [ ]:
opp_styles = ['Pal-Bot', 'Random', 'Tight', 'Limping', 'Aggressive', 'Honest-strength', 'Mixed']    # No training against "Mixed" opponents

for i in range(len(opp_styles) - 1):
    print(f"Training Q-learning agent against {opp_styles[i]} opponent...")
    train_agent(games = 500000, opponent_style = opp_styles[i], save_path=f"./data/q_tables/q_table_vs_{opp_styles[i]}.csv")
    print("\nTraining complete!")

In [ ]:
def testing_game(env, player_starts, player, opponent):
    """    
    Args:
        env: SimpleLeDucPoker instance
        player_starts: True if player acts first
        player: QLearningPlayer instance (or any player)
        opponent: PokerPlayer instance
    
    Returns:
        reward: chip change for the player
    """
    # print("\n-------------------New game started-------------------")  
    env.turn = player_starts
    # print("-------------------"+ player_starts +" starts-------------------\n")
    # print("Player card is: ", env.player_card)
    end_of_phase = False
    
    for i in range(2):
        if i == 0:
            env.initial_bet(player,opponent)
        if i == 1 and not env.folded:
            # print("Phase 2: Deal private cards and initial betting\n")
            # print("Public card is: ", env.public_card)
            end_of_phase = False
            player.raised = False
            opponent.raised = False
            
        while not end_of_phase:
            legal_actions = env.get_legal_actions(env.turn, player, opponent)
            history = ''.join(env.record)

            if env.turn == "player":
                if i == 0:  # Round 1
                    action = player.action_1(env.player_card, legal_actions, history, player_starts)
                else:       # Round 2
                    action = player.action_2(env.player_card, legal_actions, history, player_starts, env.public_card)
            else:  # opponent's turn
                if i == 0:  # Round 1
                    if isinstance(opponent, QLearningPlayer):
                        action = opponent.action_1(env.opponent_card, legal_actions, history, player_starts)
                    else:
                        action = opponent.action_1(env.opponent_card, legal_actions, player_starts)
                else:       # Round 2
                    if isinstance(opponent, QLearningPlayer):
                        action = opponent.action_2(env.player_card, legal_actions, history, player_starts, env.public_card)
                    else:
                        action = opponent.action_2(env.opponent_card, legal_actions, player_starts, env.public_card)
                
            reward, end_of_phase = env.step(action, player, opponent, player_starts)
                
            if end_of_phase:
                break
            
    # Return reward in terms of Big Blinds, not chips (Big Blind = 2 chips)    
    # print("\nFinal Reward:  ", reward/2)
    return reward/2

In [ ]:
def test_agent(games, opponent_style, load_path, trained_agent_style):
    """Use the trained agent to play against a fixed-style opponent - Sequential approach"""
    
    # Now play against the same opponent with the trained agent
    env = SimpleLeDucPoker()
    opponent = FixedOpponent(opponent_style)
    loaded_agent = QLearningPlayer(model_path=load_path, alpha=0, gamma=0, epsilon=0)  # Just greedy decisions during evaluation
    round_rewards = np.zeros(games)
    cumulative_rewards = np.zeros(games)
    
    for game_idx in range(games):  # Play specified number of games to evaluate
        player_starts = random.choice(['player', 'opponent'])
        round_rewards[game_idx] = testing_game(env, player_starts, loaded_agent, opponent)
        cumulative_rewards[game_idx] = cumulative_rewards[game_idx-1] + round_rewards[game_idx] if game_idx > 0 else round_rewards[game_idx]
        loaded_agent.reset()
        opponent.reset() 
        env.reset()
    np.savetxt(f"./data/testing/{trained_agent_style}_agent_vs_{opponent_style}.csv", round_rewards, delimiter=',')
    # print(f"\nMean reward over {games} evaluation games: {np.mean(round_rewards):.2f}")

    

    plt.figure()
    plt.title(f"{trained_agent_style} Agent's Reward vs {opponent_style} Opponent")
    plt.xlabel("Game Number")
    plt.ylabel("Cumulative Score")
    plt.plot(np.arange(1, games+1), cumulative_rewards, label="Cumulative Reward")
    plt.grid()
    plt.legend()
    plt.savefig(f'./data/images/{trained_agent_style}_agent_vs_{opponent_style}.jpg')
    plt.show()

In [ ]:
opp_styles = ['Pal-Bot', 'Random', 'Tight', 'Limping', 'Aggressive', 'Honest-strength', 'Mixed']
size = len(opp_styles)
mean_round_rewards = np.zeros((size-1, size))
for i in range(size):
    for j in range(size-1):
        print(f"\nEvaluation of agent trained vs {opp_styles[j]} opponent. Opponent style: {opp_styles[i]}")
        test_agent(games=50000, opponent_style=opp_styles[i], load_path=f"./data/q_tables/q_table_vs_{opp_styles[j]}.csv", trained_agent_style=opp_styles[j])
        mean_round_rewards[j,i] = np.mean(np.loadtxt(f"./data/testing/{opp_styles[j]}_agent_vs_{opp_styles[i]}.csv"))
        print("\nTest complete!")

print("Final mean round rewards per agent and opponent:")
print(mean_round_rewards, sep = "|")
np.savetxt(f"./data/Final_Results.csv", mean_round_rewards, "%.2f", delimiter = "\t|\t")

In [ ]:
def make_frozen_snapshot(agent):
    snap = QLearningPlayer(model_path=None, alpha=0.0, gamma=0.0, epsilon=0.0)
    snap.Q = agent.Q.copy()
    snap.N = agent.N.copy()
    snap.alpha = 0.0
    snap.epsilon = 0.0
    return snap


def train_smarter_agent(games, save_path, warmup_frac=0.15, mixed_frac=0.60, snapshot_interval=5000):
    """
    Curriculum for showdown arena:
      1) warmup vs easy fixed opponents
      2) mixed fixed-opponent training
      3) mixed fixed + frozen Q-learning opponents
    """

    player = QLearningPlayer(model_path=None, alpha=0.2, gamma=0.99, epsilon=1.0)
    env = SimpleLeDucPoker()

    players = ['player', 'opponent']
    player_cycle = cycle(players)

    cumulative_rewards = np.zeros(games)
    game_rewards = np.zeros(games)

    fixed_styles = ['Pal-Bot', 'Random', 'Tight', 'Limping', 'Aggressive', 'Honest-strength']

    q_opponent_paths = {
        'Tight': './data/q_tables/q_table_vs_Tight.csv',
        'Limping': './data/q_tables/q_table_vs_Limping.csv',
        'Honest-strength': './data/q_tables/q_table_vs_Honest-Strength.csv',
        'Random': './data/q_tables/q_table_vs_Random.csv',
        'Aggressive': './data/q_tables/q_table_vs_Aggressive.csv',
        'Pal-Bot': './data/q_tables/q_table_vs_Pal-Bot.csv',
    }

    loaded_q_opponents = {}
    for style, path in q_opponent_paths.items():
        if path is not None:
            loaded_q_opponents[style] = QLearningPlayer(model_path=path, alpha=0.0, gamma=0.0, epsilon=0.0)

    # frozen snapshot of current agent for self-play-style exposure
    frozen_snapshot = make_frozen_snapshot(player)

    warmup_end = int(warmup_frac * games)
    mixed_end = int(mixed_frac * games)

    for game_idx in range(games):

        # refresh frozen snapshot periodically
        if game_idx % snapshot_interval == 0:
            frozen_snapshot = make_frozen_snapshot(player)

        # curriculum opponent choice
        if game_idx < warmup_end:
            # warmup: easy fixed opponents only
            opponent = FixedOpponent(random.choice(['Random', 'Pal-Bot']))

        elif game_idx < mixed_end:
            # main phase: mix all fixed styles
            opponent = FixedOpponent(random.choice(fixed_styles))

        else:
            # late phase: mix fixed-trained Q opponents, fixed styles, and frozen snapshot
            r = random.random()
            if r < 0.35:
                opponent = FixedOpponent(random.choice(fixed_styles))
            elif r < 0.75:
                style = random.choice(['Tight', 'Limping', 'Honest-strength', 'Random'])
                opponent = loaded_q_opponents[style]
            else:
                opponent = frozen_snapshot

        player_starts = next(player_cycle)

        reward = training_game(env, player_starts, player, opponent)

        game_rewards[game_idx] = reward
        cumulative_rewards[game_idx] = reward if game_idx == 0 else cumulative_rewards[game_idx - 1] + reward

        player.reset()
        env.reset()

        # one epsilon decay per episode
        player.epsilon = max(0.01, 0.99999 * player.epsilon)

    player.save_model(save_path)
    np.savetxt("./data/training/Horn-y_Bot.csv", cumulative_rewards)

    plt.figure()
    plt.title("Horn-y_Bot's Reward vs Mixed Q-learning Arena")
    plt.xlabel("Game Number")
    plt.ylabel("Cumulative Score")
    plt.plot(np.arange(1, games + 1), cumulative_rewards, label="Cumulative Reward")
    plt.grid()
    plt.legend()
    plt.savefig('./data/images/Horn-y_Bot.jpg')
    plt.show()

    return player, game_rewards, cumulative_rewards

In [ ]:
print(f"Training smart Q-learning agent against Q-learning opponents...")
train_smarter_agent(games = 500000, save_path = f"./data/q_tables/Horn-y_Bot_table.csv")
print("\nTraining complete!")

In [ ]:
def test_smart_agent(games, load_player_path, load_opp_path, trained_agent_style):
    """Use the trained agent to play against a fixed-style opponent - Sequential approach"""
    
    # Now play against the same opponent with the trained agent
    env = SimpleLeDucPoker()
    opponent = QLearningPlayer(model_path=load_opp_path, alpha=0, gamma=0, epsilon=0)
    player = QLearningPlayer(model_path=load_player_path, alpha=0, gamma=0, epsilon=0)  # Just greedy decisions during evaluation
    round_rewards = np.zeros(games)
    cumulative_rewards = np.zeros(games)
    
    for game_idx in range(games):  # Play specified number of games to evaluate
        player_starts = random.choice(['player', 'opponent'])
        round_rewards[game_idx] = testing_game(env, player_starts, player, opponent)
        cumulative_rewards[game_idx] = cumulative_rewards[game_idx-1] + round_rewards[game_idx] if game_idx > 0 else round_rewards[game_idx]
        player.reset()  
        opponent.reset() 
        env.reset()
    np.savetxt(f"./data/testing/Horn-y_Bot_vs_{trained_agent_style}_agent.csv", round_rewards, delimiter=',')
    # print(f"\nMean reward over {games} evaluation games: {np.mean(round_rewards):.2f}")

    plt.figure()
    plt.title(f"Horn-y Bot's Reward vs {trained_agent_style} Agent")
    plt.xlabel("Game Number")
    plt.ylabel("Cumulative Score")
    plt.plot(np.arange(1, games+1), cumulative_rewards, label="Cumulative Reward")
    plt.grid()
    plt.legend()
    plt.savefig(f'./data/images/Horn-y_Bot_vs_{trained_agent_style}_agent.jpg')
    plt.show()

In [ ]:
opp_training_styles = ['Tight', 'Limping', 'Honest-strength']
size = len(opp_training_styles)
mean_round_rewards = np.zeros(size)

for i in range(size):
    print(f"\nEvaluation of Horn-y Bot vs {opp_training_styles[i]}-trained agent.")
    test_smart_agent(games=50000, load_player_path=f"./data/q_tables/Horn-y_Bot_table.csv", load_opp_path=f"./data/q_tables/q_table_vs_{opp_training_styles[i]}.csv", trained_agent_style=opp_training_styles[i])
    test_agent(games=50000, opponent_style=opp_training_styles[i], load_path=f"./data/q_tables/Horn-y_Bot_table.csv", trained_agent_style=opp_training_styles[i])
    mean_round_rewards[i] = np.mean(np.loadtxt(f"./data/testing/Horn-y_Bot_vs_{opp_training_styles[i]}_agent.csv"))
    print("\nTest complete!")

print("Final mean round rewards per agent and opponent:")
print(mean_round_rewards, sep = "|")
np.savetxt(f"./data/Horn-y_Bot.csv", mean_round_rewards, "%.2f", delimiter = "|")

In [ ]:
def train_meta_agent(games, save_path, horny_bot_path,
                     warmup_frac=0.10, rampup_frac=0.35,
                     snapshot_interval=5000):
    """
    Train a new agent whose primary sparring partner is Horn-y Bot
    (itself trained via curriculum against all specialist Q-agents).

    Curriculum
    ----------
    Phase 1 – warmup   (0   → 10 %): easy fixed opponents only
    Phase 2 – ramp-up  (10  → 35 %): all fixed styles + specialist Q-agents
    Phase 3 – main     (35 → 100 %): 50 % Horn-y Bot | 20 % specialists
                                    | 15 % fixed styles | 15 % frozen self-play
    """
    player = QLearningPlayer(model_path=None, alpha=0.2, gamma=0.99, epsilon=1.0)
    env    = SimpleLeDucPoker()
    sides  = cycle(['player', 'opponent'])

    game_rewards       = np.zeros(games)
    cumulative_rewards = np.zeros(games)

    fixed_styles = ['Pal-Bot', 'Random', 'Tight', 'Limping', 'Aggressive', 'Honest-strength']

    # Load Horn-y Bot – the main training rival
    horny_bot = QLearningPlayer(model_path=horny_bot_path,
                                alpha=0.0, gamma=0.0, epsilon=0.0)

    # Load every specialist Q-agent
    # Note: filenames use lowercase 's' in 'Honest-strength' (matching train_agent output)
    specialist_paths = {s: f'./data/q_tables/q_table_vs_{s}.csv' for s in fixed_styles}
    specialists = {
        s: QLearningPlayer(model_path=p, alpha=0.0, gamma=0.0, epsilon=0.0)
        for s, p in specialist_paths.items()
    }

    frozen_snapshot = make_frozen_snapshot(player)
    warmup_end  = int(warmup_frac  * games)
    rampup_end  = int(rampup_frac  * games)

    for game_idx in range(games):

        # Refresh frozen self-play snapshot periodically
        if game_idx % snapshot_interval == 0:
            frozen_snapshot = make_frozen_snapshot(player)

        # ── Curriculum opponent selection ──────────────────────────── #
        if game_idx < warmup_end:
            # Phase 1: easy opponents to bootstrap Q-values
            opponent = FixedOpponent(random.choice(['Random', 'Pal-Bot', 'Limping']))

        elif game_idx < rampup_end:
            # Phase 2: broaden exposure before meeting Horn-y Bot
            r = random.random()
            if r < 0.55:
                opponent = FixedOpponent(random.choice(fixed_styles))
            else:
                opponent = specialists[random.choice(list(specialists))]

        else:
            # Phase 3: target Horn-y Bot with supporting variety
            r = random.random()
            if   r < 0.50: opponent = horny_bot                                      # 50 % Horn-y Bot
            elif r < 0.70: opponent = specialists[random.choice(list(specialists))]  # 20 % specialists
            elif r < 0.85: opponent = FixedOpponent(random.choice(fixed_styles))     # 15 % fixed styles
            else:          opponent = frozen_snapshot                                 # 15 % self-play

        player_starts = next(sides)
        reward        = training_game(env, player_starts, player, opponent)

        game_rewards[game_idx]       = reward
        cumulative_rewards[game_idx] = (reward if game_idx == 0
                                        else cumulative_rewards[game_idx - 1] + reward)

        player.reset()
        opponent.reset()
        env.reset()

        player.epsilon = max(0.01, 0.99999 * player.epsilon)

    player.save_model(save_path)
    np.savetxt('./data/training/meta_agent.csv', cumulative_rewards)

    # Smoothed learning curve for readability
    window = max(1, games // 200)
    smoothed = np.convolve(game_rewards, np.ones(window) / window, mode='valid')

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(np.arange(1, games + 1), cumulative_rewards, color='#2ecc71', linewidth=0.7)
    axes[0].set_title('Meta-Agent – Cumulative Reward', fontweight='bold')
    axes[0].set_xlabel('Game')
    axes[0].set_ylabel('Cumulative BB')
    axes[0].grid(alpha=0.3)

    axes[1].plot(np.arange(1, len(smoothed) + 1), smoothed, color='#e67e22', linewidth=0.9)
    axes[1].axhline(0, color='black', linewidth=0.6, linestyle='--')
    axes[1].set_title(f'Meta-Agent – Rolling Avg Reward (window={window})', fontweight='bold')
    axes[1].set_xlabel('Game')
    axes[1].set_ylabel('Avg BB / hand')
    axes[1].grid(alpha=0.3)

    plt.suptitle('Meta-Agent Training vs Horn-y Bot Arena', fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig('./data/images/meta_agent_training.jpg', dpi=150, bbox_inches='tight')
    plt.show()

    return player, game_rewards, cumulative_rewards

In [ ]:
print("Training Meta-Agent vs Horn-y Bot…")
meta_agent, meta_game_rewards, meta_cumul_rewards = train_meta_agent(
    games          = 500_000,
    save_path      = './data/q_tables/meta_agent_table.csv',
    horny_bot_path = './data/q_tables/Horn-y_Bot_table.csv',
)
print("Training complete!")

In [ ]:
GAMES_PER_MATCHUP = 10_000   # ← reduce (e.g. 10_000) for a faster run

# ─── Universal testing game ─────────────────────────────────────────────
# Handles any combination of QLearningPlayer / FixedOpponent in either slot.
# Mirrors the exact branch logic of testing_game() for backward-compatibility
# (including the intentional player_card/opponent_card asymmetry for Q-agents
#  in the opponent slot during round 2, which was present during their training).

def universal_testing_game(env, player_starts, agent_a, agent_b):
    """One game. agent_a = player role, agent_b = opponent role.
    Returns BB reward from agent_a's perspective."""
    env.turn     = player_starts
    end_of_phase = False

    for round_idx in range(2):
        if round_idx == 0:
            env.initial_bet(agent_a, agent_b)
        elif not env.folded:
            end_of_phase  = False
            agent_a.raised = False
            agent_b.raised = False

        while not end_of_phase:
            legal   = env.get_legal_actions(env.turn, agent_a, agent_b)
            history = ''.join(env.record)

            if env.turn == "player":                          # ── agent_a ──
                if isinstance(agent_a, QLearningPlayer):
                    action = (agent_a.action_1(env.player_card, legal, history, player_starts)
                              if round_idx == 0 else
                              agent_a.action_2(env.player_card, legal, history, player_starts, env.public_card))
                else:                                         # FixedOpponent
                    action = (agent_a.action_1(env.player_card, legal, player_starts)
                              if round_idx == 0 else
                              agent_a.action_2(env.player_card, legal, player_starts, env.public_card))

            else:                                             # ── agent_b ──
                if isinstance(agent_b, QLearningPlayer):
                    # Round-2 intentionally passes player_card (matches training behaviour)
                    action = (agent_b.action_1(env.opponent_card, legal, history, player_starts)
                              if round_idx == 0 else
                              agent_b.action_2(env.player_card, legal, history, player_starts, env.public_card))
                else:                                         # FixedOpponent
                    action = (agent_b.action_1(env.opponent_card, legal, player_starts)
                              if round_idx == 0 else
                              agent_b.action_2(env.opponent_card, legal, player_starts, env.public_card))

            reward, end_of_phase = env.step(action, agent_a, agent_b, player_starts)
            if end_of_phase:
                break

    return reward / 2

In [ ]:
def run_matchup(agent_a, agent_b, games):
    """Round-robin matchup (alternating sides). Returns mean BB/hand for agent_a."""
    env     = SimpleLeDucPoker()
    rewards = np.zeros(games)
    sides   = cycle(['player', 'opponent'])
    for g in range(games):
        player_starts = next(sides)
        rewards[g]    = universal_testing_game(env, player_starts, agent_a, agent_b)
        agent_a.reset()
        agent_b.reset()
        env.reset()
    return float(np.mean(rewards))


In [ ]:
fixed_styles = ['Pal-Bot', 'Random', 'Tight', 'Limping', 'Aggressive', 'Honest-strength']

all_agents = {}

# Fixed-style baselines
for s in fixed_styles:
    all_agents[s] = FixedOpponent(s)

# Specialist Q-agents (one per fixed style, trained against it)
for s in fixed_styles:
    all_agents[f'Q-{s}'] = QLearningPlayer(
        model_path=f'./data/q_tables/q_table_vs_{s}.csv',
        alpha=0, gamma=0, epsilon=0)

# Horn-y Bot (curriculum-trained vs all specialists)
all_agents['Horn-y Bot'] = QLearningPlayer(
    model_path='./data/q_tables/Horn-y_Bot_table.csv',
    alpha=0, gamma=0, epsilon=0)

# Meta-Agent (trained against Horn-y Bot as main rival)
all_agents['Meta-Agent ⭐'] = QLearningPlayer(
    model_path='./data/q_tables/meta_agent_table.csv',
    alpha=0, gamma=0, epsilon=0)

agent_names = list(all_agents.keys())
n           = len(agent_names)

# ─── Round-robin tournament ──────────────────────────────────────────────
results = np.zeros((n, n))

print(f"Running {n}×{n} round-robin tournament " f"({GAMES_PER_MATCHUP:,} games / matchup)…\n")

for i, a_name in enumerate(agent_names):
    for j, b_name in enumerate(agent_names):
        if i == j:
            results[i, j] = 0.0
            continue
        r = run_matchup(all_agents[a_name], all_agents[b_name], GAMES_PER_MATCHUP)
        results[i, j] = r
        print(f"  {a_name:24s} vs {b_name:24s}  →  {r:+.4f}")

# ─── Leaderboard (average across all opponents) ──────────────────────────
off_diag = ~np.eye(n, dtype=bool)
scores   = np.where(off_diag, results, np.nan)
avg_score = np.nanmean(scores, axis=1)
ranking   = np.argsort(-avg_score)

print("\n" + "═" * 62)
print(f"{'  GRAND TOURNAMENT LEADERBOARD':^62}")
print("═" * 62)
print(f"  {'Rank':<5}  {'Agent':<28}  {'Avg BB/hand':>11}")
print("─" * 62)
medals = {1: "🥇", 2: "🥈", 3: "🥉"}
for rank, idx in enumerate(ranking, 1):
    m = medals.get(rank, "  ")
    print(f"  {m} {rank:<3}  {agent_names[idx]:<28}  {avg_score[idx]:>+.4f}")
print("═" * 62)

# ─── Figure 1: Heatmap ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(17, 12))
im = ax.imshow(results, cmap='RdYlGn', aspect='auto', vmin=-0.5, vmax=0.5)

ax.set_xticks(np.arange(n))
ax.set_yticks(np.arange(n))
ax.set_xticklabels(agent_names, rotation=40, ha='right', fontsize=8.5)
ax.set_yticklabels(agent_names, fontsize=8.5)

# Bold border around Meta-Agent row and column
for spine in ['top', 'bottom', 'left', 'right']:
    ax.spines[spine].set_linewidth(1.5)

for i in range(n):
    for j in range(n):
        val   = results[i, j]
        color = 'white' if abs(val) > 0.25 else 'black'
        ax.text(j, i, f'{val:+.3f}', ha='center', va='center', fontsize=7, color=color, fontweight='bold')

plt.colorbar(im, ax=ax, label='Mean BB / hand  (positive → row agent wins)')
ax.set_title('Grand Competition – Head-to-Head Mean BB/Hand\n' '(green = row agent wins | Meta-Agent highlighted in green border)', fontsize=12, fontweight='bold', pad=14)
ax.set_xlabel('Opponent →', fontsize=10)
ax.set_ylabel('← Player',  fontsize=10)
plt.tight_layout()
plt.savefig('./data/images/grand_competition_heatmap.jpg', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
#!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!---FOR BOT EXTRACTION---!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

In [ ]:
import numpy as np
import json
from pathlib import Path

# ──────────────────────────────────────────────────────────────────────────────
# RLCard action order:
#   0 = call, 1 = raise, 2 = fold, 3 = check
#
# My old action order:
#   0 = check, 1 = call, 2 = raise, 3 = fold
#
#   RLCard[0=call]  = my[1]
#   RLCard[1=raise] = my[2]
#   RLCard[2=fold]  = my[3]
#   RLCard[3=check] = my[0]
# ──────────────────────────────────────────────────────────────────────────────
REORDER = [1, 2, 3, 0]   # your indices that map to RLCard positions 0,1,2,3 
# ──────────────────────────────────────────────────────────────────────────────
# Maps (history, blind, is_round2) → (my_chips, opp_chips)
# Derived by simulating the LeDuc betting rules:
#   Round 1: raise = call_diff + 2
#   Round 2: raise = call_diff + 4
# ──────────────────────────────────────────────────────────────────────────────
CHIPS_MAP = {
    # (history, blind, is_round2) : (my_chips, opp_chips)
    ('',    'SB', False): (1,  2),
    ('',    'SB', True):  (1,  2),   # round-2 empty history = just entered round 2
    ('',    'BB', False): (2,  1),
    ('',    'BB', True):  (2,  1),
 
    ('c',   'SB', False): (2,  2),
    ('c',   'SB', True):  (2,  2),
    ('c',   'BB', False): (2,  2),
    ('c',   'BB', True):  (2,  2),
 
    ('r',   'SB', False): (4,  2),
    ('r',   'SB', True):  (6,  2),
    ('r',   'BB', False): (2,  4),
    ('r',   'BB', True):  (2,  6),
 
    ('cr',  'SB', False): (2,  4),
    ('cr',  'SB', True):  (2,  6),
    ('cr',  'BB', False): (4,  2),
    ('cr',  'BB', True):  (6,  2),
 
    ('rr',  'SB', False): (4,  6),
    ('rr',  'SB', True):  (6, 10),
    ('rr',  'BB', False): (6,  4),
    ('rr',  'BB', True):  (10, 6),
 
    ('crr', 'SB', False): (6,  4),
    ('crr', 'SB', True):  (10, 6),
    ('crr', 'BB', False): (4,  6),
    ('crr', 'BB', True):  (6, 10),
}

def decode_my_state(state_idx):
    """
    Reverse your encode_state() to recover (private_card, public_card, history, blind).
    Your encoding:
        state = (private_idx * 4 * 6 * 2)
                + (public_idx  *     6 * 2)
                + (history_idx *         2)
                + blind_idx
    """
    idx_to_card    = {0: 'J', 1: 'Q', 2: 'K'}
    idx_to_public  = {0: None, 1: 'J', 2: 'Q', 3: 'K'}
    idx_to_history = {0: '', 1: 'c', 2: 'r', 3: 'cr', 4: 'rr', 5: 'crr'}
    idx_to_blind   = {0: 'SB', 1: 'BB'}

    blind_idx   =  state_idx % 2
    history_idx = (state_idx // 2) % 6
    public_idx  = (state_idx // 12) % 4
    private_idx = (state_idx // 48) % 3

    return (
        idx_to_card[private_idx],
        idx_to_public[public_idx],
        idx_to_history[history_idx],
        idx_to_blind[blind_idx],
    )

def build_rlcard_obs(private_card, public_card, my_chips, opp_chips):
    """
    Reconstruct the RLCard obs vector (length 36) from game info.
    RLCard encoding (from LeducholdemEnv._extract_state):
        obs[card2index[hand]]            = 1   # positions 0-2  (J=0, Q=1, K=2)
        obs[card2index[public_card] + 3] = 1   # positions 3-5  (or none if round 1)
        obs[my_chips + 6]                = 1   # positions 7-20
        obs[opp_chips + 21]              = 1   # positions 22-35
    """
    card2index = {'J': 0, 'Q': 1, 'K': 2}

    obs = np.zeros(36, dtype=np.float64)
    obs[card2index[private_card]] = 1.0

    if public_card is not None:
        obs[card2index[public_card] + 3] = 1.0

    obs[my_chips + 6]  = 1.0
    obs[opp_chips + 21] = 1.0

    return obs

def convert_q_table(q_table_csv_path, output_npz_path=None):
    """
    Convert your trained Q-table (144×4 numpy array saved as CSV)
    into the RLCard-compatible format: dict { tuple(obs): np.array([q_call, q_raise, q_fold, q_check]) }
    and save it as q_table.npz.

    Args:
        q_table_csv_path : path to your CSV file (e.g. 'q_table_vs_Aggressive.csv')
        output_npz_path  : where to save the .npz (default: same folder, 'q_table.npz')

    Returns:
        Q_dict : the converted dictionary (also saved to disk)
    """
    q_table_csv_path = Path(q_table_csv_path)
    if output_npz_path is None:
        output_npz_path = q_table_csv_path.parent / "q_table.npz"
    output_npz_path = Path(output_npz_path)

    Q = np.loadtxt(q_table_csv_path, delimiter=',')
    assert Q.shape == (144, 4), f"Expected shape (144, 4), got {Q.shape}"

    Q_dict = {}          # state_key (tuple of ints) → q-values (4,)
    collision_log = []   # states that share the same obs key

    for state_idx in range(144):
        private_card, public_card, history, blind = decode_my_state(state_idx)

        is_round2 = public_card is not None
        chips_key = (history, blind, is_round2)

        if chips_key not in CHIPS_MAP:
            print(f"[WARN] No chips mapping for state {state_idx}: {chips_key}")
            continue

        my_chips, opp_chips = CHIPS_MAP[chips_key]
        obs = build_rlcard_obs(private_card, public_card, my_chips, opp_chips)
        state_key = tuple(int(x) for x in obs)

        # Reorder q-values from your order [check,call,raise,fold] to RLCard order [call, raise, fold, check]
        q_yours   = Q[state_idx]                    
        q_rlcard  = q_yours[REORDER]                
        
        if state_key in Q_dict:
            # Multiple of your states map to the same RLCard obs key.
            # Keep the one with higher max Q-value (most confident).
            if np.max(q_rlcard) > np.max(Q_dict[state_key]):
                Q_dict[state_key] = q_rlcard
            collision_log.append((state_idx, private_card, public_card, history, blind))
        else:
            Q_dict[state_key] = q_rlcard

    if collision_log:
        print(f"[INFO] {len(collision_log)} states shared an obs key with another state (kept the one with highest max Q). Affected states:")
        for s in collision_log:
            print(f"       state {s[0]}: private={s[1]}, public={s[2]}, "
                f"history='{s[3]}', blind={s[4]}")

    keys   = np.asarray([json.dumps(list(k)) for k in Q_dict.keys()], dtype=object)
    values = np.asarray(list(Q_dict.values()), dtype=np.float64)

    np.savez(output_npz_path, keys=keys, values=values)
    print(f"[OK] Saved {len(Q_dict)} states to '{output_npz_path}'")

    return Q_dict

In [ ]:
import json
import re
import shutil
from pathlib import Path

import numpy as np


# ============================================================
# USER SETTINGS — EDIT ONLY THIS SECTION
# ============================================================

GROUP_NUMBER = 17
AGENT_NICKNAME = "Horn-y Bot"

# Examples:
# Q_AGENT = q_agent
# Q_AGENT = trained_agent
# Q_AGENT = strategy3_result["agent"]
# Q_AGENT = my_Q_dictionary
Q_AGENT = convert_q_table("./data/q_tables/Horn-y_Bot_table.csv")


TEAM_MEMBERS = [
    "Pavlos Horn",
    "Giorgos Vogiatzis",
]

DESCRIPTION = "Tabular Q-learning agent trained for RLCard Leduc Hold'em."

TRAINING_DETAILS = {
    "training_method": "Tabular Q-learning",
    "training_opponents": "Q-learning opponent agents trained to beat fixed 'Limping', 'Tight', and 'Honest-Strength' playing styles",
    "num_training_episodes": "500000",
    "alpha_schedule": "alpha = max(0.01, 20 / [20 + N(s,a)]), N(s,a): the number of updates of q(s,a) in the Q-table ",
    "epsilon_schedule": "epsilon(t) = max{0.99999^t, 0.01}, epsilon(0) = 1",
    "discount_factor": "gamma = 0.99",
    "state_encoding": "tuple(int(x) for x in state['obs'])",
    "comments": "Add any other useful information",
}

# The generated folder and zip file will be written here.
# To save directly to Google Drive in Colab, mount Drive and use, for example:
# SUBMISSION_ROOT = Path("/content/drive/MyDrive/leduc_agent_submissions")
SUBMISSION_ROOT = Path("leduc_agent_submissions")


# ============================================================
# DO NOT EDIT BELOW
# ============================================================

AGENT_PY_TEMPLATE = r"""
import json
import random
from pathlib import Path

import numpy as np


ACTION_NAMES = {
    0: "call",
    1: "raise",
    2: "fold",
    3: "check",
}


def legal_action_ids(state):
    return list(state["legal_actions"].keys())


def encode_state_from_obs(state):
    return tuple(int(x) for x in state["obs"])


def argmax_over_legal(q_values, legal_actions, rng):
    best_value = max(q_values[a] for a in legal_actions)
    best_actions = [a for a in legal_actions if q_values[a] == best_value]
    return rng.choice(best_actions)


class PokerAgent:
    \"\"\"Non-adaptive Leduc agent loaded from a submitted Q-table.\"\"\"

    def __init__(self, model_dir=None, seed=None):
        self.rng = random.Random(seed)
        self.model_dir = Path("." if model_dir is None else model_dir)

        q_path = self.model_dir / "q_table.npz"
        if not q_path.exists():
            raise FileNotFoundError(f"Could not find q_table.npz at {q_path}")

        data = np.load(q_path, allow_pickle=True)

        self.Q = {}
        for key_str, q_values in zip(data["keys"], data["values"]):
            state_key = tuple(json.loads(str(key_str)))
            self.Q[state_key] = np.asarray(q_values, dtype=np.float64)

        metadata_path = self.model_dir / "metadata.json"
        if metadata_path.exists():
            with open(metadata_path, "r", encoding="utf-8") as f:
                self.metadata = json.load(f)
        else:
            self.metadata = {}

        self.name = (
            self.metadata.get("agent_name")
            or self.metadata.get("team_name")
            or self.model_dir.name
        )

    def act(self, state):
        legal_actions = legal_action_ids(state)
        if not legal_actions:
            raise ValueError("RLCard returned a state with no legal actions.")

        state_key = encode_state_from_obs(state)

        if state_key not in self.Q:
            return self.rng.choice(legal_actions)

        q_values = self.Q[state_key]
        return argmax_over_legal(q_values, legal_actions, self.rng)
"""


def make_safe_name(name):
    """Convert a nickname to a safe folder/file component."""
    name = str(name).strip()
    name = re.sub(r"[^A-Za-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name)
    return name.strip("_")


def extract_q_table(q_agent_or_q_dict):
    """Accept either a raw Q dictionary or an agent object with attribute .Q."""
    if isinstance(q_agent_or_q_dict, dict):
        return q_agent_or_q_dict

    if hasattr(q_agent_or_q_dict, "Q"):
        return q_agent_or_q_dict.Q

    raise TypeError(
        "Q_AGENT must be either a Q-table dictionary or an object with a .Q attribute."
    )


def save_python_q_agent_submission(
    q_agent_or_q_dict,
    output_dir,
    agent_name,
    group_number,
    agent_nickname,
    team_members,
    description,
    training_details,
):
    """Save agent.py, q_table.npz, metadata.json, and README.txt."""

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    Q = extract_q_table(q_agent_or_q_dict)

    if len(Q) == 0:
        raise ValueError("The supplied Q-table is empty.")

    keys = []
    values = []

    for state_key, q_values in Q.items():
        try:
            key_as_list = [int(x) for x in state_key]
        except Exception as exc:
            raise TypeError(
                f"State key {state_key!r} cannot be converted to a tuple/list of integers."
            ) from exc

        q_values = np.asarray(q_values, dtype=np.float64)

        if q_values.shape != (4,):
            raise ValueError(
                f"Q-values for state {state_key!r} must have shape (4,), "
                f"but got {q_values.shape}."
            )

        if not np.all(np.isfinite(q_values)):
            raise ValueError(f"Q-values for state {state_key!r} contain NaN or infinity.")

        keys.append(json.dumps(key_as_list))
        values.append(q_values)

    np.savez(
        output_dir / "q_table.npz",
        keys=np.asarray(keys, dtype=object),
        values=np.asarray(values, dtype=np.float64),
    )

    with open(output_dir / "agent.py", "w", encoding="utf-8") as f:
        f.write(AGENT_PY_TEMPLATE.strip() + "\n")

    metadata = {
        "agent_name": agent_name,
        "team_name": agent_name,
        "group_number": int(group_number),
        "agent_nickname": agent_nickname,
        "team_members": list(team_members),
        "description": description,
        "num_q_states": int(len(keys)),
        "submission_type": "python_agent_plus_q_table",
        "entry_file": "agent.py",
        "entry_class": "PokerAgent",
        "model_file": "q_table.npz",
        "action_order": {
            "0": "call",
            "1": "raise",
            "2": "fold",
            "3": "check",
        },
        "state_encoding": "tuple(int(x) for x in state['obs'])",
        "policy_at_evaluation": "greedy argmax over legal actions using submitted Q-table",
        "fallback_for_unseen_states": "random legal action",
        "non_adaptive": True,
        "training_details": training_details,
    }

    with open(output_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    with open(output_dir / "README.txt", "w", encoding="utf-8") as f:
        f.write(f"Agent name: {agent_name}\n")
        f.write(f"Group number: {group_number}\n")
        f.write(f"Agent nickname: {agent_nickname}\n")
        f.write(f"Team members: {', '.join(team_members)}\n")
        f.write(f"Description: {description}\n")
        f.write(f"Number of Q-table states: {len(keys)}\n")

        f.write("\nFiles:\n")
        f.write("  agent.py      Python wrapper defining PokerAgent\n")
        f.write("  q_table.npz   Saved Q-table\n")
        f.write("  metadata.json Submission metadata\n")
        f.write("  README.txt    Submission summary\n")

        f.write("\nEvaluation policy:\n")
        f.write("  Encode state['obs'] as a tuple of integers.\n")
        f.write("  Look up Q-values and select the greedy legal action.\n")
        f.write("  Break ties randomly.\n")
        f.write("  Use a random legal action for unseen states.\n")

        f.write("\nTraining details:\n")
        for key, value in training_details.items():
            f.write(f"  {key}: {value}\n")

    return output_dir


# ------------------------------------------------------------
# Validate user settings
# ------------------------------------------------------------

if Q_AGENT is None:
    raise ValueError(
        "Set Q_AGENT in the USER SETTINGS section to your trained agent "
        "or directly to your Q-table dictionary before running this cell."
    )

if not isinstance(GROUP_NUMBER, (int, np.integer)) or int(GROUP_NUMBER) < 0:
    raise ValueError("GROUP_NUMBER must be a non-negative integer.")

safe_nickname = make_safe_name(AGENT_NICKNAME)
if not safe_nickname:
    raise ValueError("AGENT_NICKNAME must contain at least one letter or number.")

agent_name = f"LeDuc_{int(GROUP_NUMBER)}_{safe_nickname}"
SUBMISSION_ROOT.mkdir(parents=True, exist_ok=True)
output_dir = SUBMISSION_ROOT / agent_name

# Remove an old export with the same name to avoid stale files.
if output_dir.exists():
    shutil.rmtree(output_dir)

save_python_q_agent_submission(
    q_agent_or_q_dict=Q_AGENT,
    output_dir=output_dir,
    agent_name=agent_name,
    group_number=int(GROUP_NUMBER),
    agent_nickname=AGENT_NICKNAME,
    team_members=TEAM_MEMBERS,
    description=DESCRIPTION,
    training_details=TRAINING_DETAILS,
)

# Create LeDuc_<group>_<nickname>.zip containing the matching folder.
zip_base = SUBMISSION_ROOT / agent_name
zip_file = Path(str(zip_base) + ".zip")
if zip_file.exists():
    zip_file.unlink()

zip_path = shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(SUBMISSION_ROOT),
    base_dir=agent_name,
)

print()
print("Submission created successfully.")
print(f"Agent name:        {agent_name}")
print(f"Number of states:  {len(extract_q_table(Q_AGENT))}")
print(f"Submission folder: {output_dir.resolve()}")
print(f"Submission zip:    {Path(zip_path).resolve()}")
print()
print("Submit this file:")
print(f"  {Path(zip_path).name}")


In [ ]:
sys.path.insert(0, '.')
RLCARD_ACTIONS = ['call', 'raise', 'fold', 'check']

def inspect(csv_path):
    Q = np.loadtxt(csv_path, delimiter=',')
    assert Q.shape == (144, 4), f"Expected (144,4), got {Q.shape}"

    # First pass: build all rows
    rows = []
    for state_idx in range(144):
        private, public, history, blind = decode_my_state(state_idx)
        is_round2 = public is not None
        my_chips, opp_chips = CHIPS_MAP[(history, blind, is_round2)]
        obs = build_rlcard_obs(private, public, my_chips, opp_chips)
        state_key = tuple(int(x) for x in obs)
        q_yours  = Q[state_idx]
        q_rlcard = q_yours[REORDER]   # reorder to [call, raise, fold, check]
        best_action = RLCARD_ACTIONS[int(np.argmax(q_rlcard))]
        rows.append({
            'idx':       state_idx,
            'private':   private,
            'public':    public or 'None',
            'history':   f"'{history}'",
            'blind':     blind,
            'my_chips':  my_chips,
            'opp_chips': opp_chips,
            'q_call':    q_rlcard[0],
            'q_raise':   q_rlcard[1],
            'q_fold':    q_rlcard[2],
            'q_check':   q_rlcard[3],
            'maxQ':      float(np.max(q_rlcard)),
            'best':      best_action,
            'key':       state_key,
            'winner':    True,   # assume winner until proven otherwise
        })

    # Second pass: for each obs key, keep only the row with highest maxQ
    # Group by key
    from collections import defaultdict
    key_to_rows = defaultdict(list)
    for r in rows:
        key_to_rows[r['key']].append(r)

    for key, group in key_to_rows.items():
        if len(group) > 1:
            best = max(group, key=lambda r: r['maxQ'])
            for r in group:
                if r is not best:
                    r['winner'] = False
                    r['beaten_by'] = best['idx']
            best['beaten_by'] = None

    # ── Print header ──────────────────────────────────────────────────────────
    sep = "-" * 135
    hdr = (f"{'#':>4}  {'Priv':>4}  {'Pub':>4}  {'Hist':>5}  {'Blind':>4}  "
            f"{'MyCh':>4}  {'OppCh':>5}  "
            f"{'q_call':>8}  {'q_raise':>8}  {'q_fold':>8}  {'q_check':>8}  "
            f"{'BEST':>6}  NOTE")
    print(sep)
    print(hdr)
    print(sep)

    for r in rows:
        if r['winner']:
            marker = "✓"
            note = ""
        else:
            marker = "✗"
            note = f"DROPPED — state {r['beaten_by']} kept (higher max Q = {max(rr['maxQ'] for rr in key_to_rows[r['key']]):.4f})"

        print(
            f"{r['idx']:>4}  {r['private']:>4}  {r['public']:>4}  {r['history']:>5}  {r['blind']:>4}  "
            f"{r['my_chips']:>4}  {r['opp_chips']:>5}  "
            f"{r['q_call']:>8.4f}  {r['q_raise']:>8.4f}  {r['q_fold']:>8.4f}  {r['q_check']:>8.4f}  "
            f"{r['best']:>6}  {marker} {note}"
        )

    # ── Summary ───────────────────────────────────────────────────────────────
    n_dropped = sum(1 for r in rows if not r['winner'])
    n_unique  = 144 - n_dropped
    print(sep)
    print(f"Total your states : 144")
    print(f"Unique RLCard keys: {n_unique}  (dropped: {n_dropped})")
    print(sep)

    # ── Per-card best action summary ──────────────────────────────────────────
    print("\n=== Best action per (private, public, history) [only KEPT states] ===")
    print(f"{'Private':>7}  {'Public':>6}  {'History':>7}  {'Best (SB)':>10}  {'Best (BB)':>10}  NOTE")
    print("-" * 70)
    for priv in ['J', 'Q', 'K']:
        for pub in [None, 'J', 'Q', 'K']:
            for hist in ['', 'c', 'r', 'cr', 'rr', 'crr']:
                pub_str = pub or 'None'
                sb = next((r for r in rows if r['private']==priv and r['public']==pub_str and r['history']==f"'{hist}'" and r['blind']=='SB'), None)
                bb = next((r for r in rows if r['private']==priv and r['public']==pub_str and r['history']==f"'{hist}'" and r['blind']=='BB'), None)
                sb_best = (sb['best'] if sb['winner'] else f"({sb['best']})") if sb else '-'
                bb_best = (bb['best'] if bb['winner'] else f"({bb['best']})") if bb else '-'
                note = ""
                if sb and not sb['winner']: note += f"SB dropped→{sb['beaten_by']} "
                if bb and not bb['winner']: note += f"BB dropped→{bb['beaten_by']}"
                print(f"{priv:>7}  {pub_str:>6}  {hist!r:>7}  {sb_best:>10}  {bb_best:>10}  {note}")


inspect("./data/q_tables/Horn-y_Bot_table.csv")

In [ ]:
import numpy as np
import json

data = np.load("./leduc_agent_submissions/LeDuc_17_Horn_y_Bot/q_table.npz", allow_pickle=True)

print(f"{'State key (obs tuple)':>50}  {'q_call':>8}  {'q_raise':>8}  {'q_fold':>8}  {'q_check':>8}  {'BEST':>6}")
print("-" * 105)

action_names = ['call', 'raise', 'fold', 'check']
for key_str, q_vals in zip(data["keys"], data["values"]):
    key = tuple(json.loads(str(key_str)))
    best = action_names[int(np.argmax(q_vals))]
    print(f"{str(key):>50}  {q_vals[0]:>8.4f}  {q_vals[1]:>8.4f}  {q_vals[2]:>8.4f}  {q_vals[3]:>8.4f}  {best:>6}")

print(f"\nTotal states in npz: {len(data['keys'])}")